In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
print("✅ Environment ready!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ Environment ready!
Pandas version: 2.3.3
NumPy version: 2.4.1


In [ ]:
print("Loading PATIENTS.csv...")
patients = pd.read_csv('../../data/raw/PATIENTS.csv')

print(f"\n✅ Loaded {len(patients):,} patients")
print(f"\nFirst 5 rows:")
print(patients.head())

print(f"\nColumns: {patients.columns.tolist()}")
print(f"\nData types:")
print(patients.dtypes)

print(f"\nGender distribution:")
print(patients['GENDER'].value_counts())

print(f"\nMissing values:")
print(patients.isnull().sum())

In [ ]:
# Load DIAGNOSES_ICD table
print("Loading DIAGNOSES_ICD.csv...")
diagnoses = pd.read_csv('../../data/raw/DIAGNOSES_ICD.csv')

print(f"\n✅ Loaded {len(diagnoses):,} diagnosis records")
print(f"\nFirst 10 rows:")
print(diagnoses.head(10))

print(f"\nColumns: {diagnoses.columns.tolist()}")

print(f"\nUnique patients with diagnoses: {diagnoses['SUBJECT_ID'].nunique():,}")
print(f"Unique ICD9 codes: {diagnoses['ICD9_CODE'].nunique():,}")

In [5]:
diagnosis_counts = diagnoses['ICD9_CODE'].value_counts()

print("📊 Top 20 most common diagnoses:\n")
print(diagnosis_counts.head(20))

# Save top codes
top_20_codes = diagnosis_counts.head(20).index.tolist()
print(f"\n✅ Top 20 ICD9 codes saved: {top_20_codes}")

📊 Top 20 most common diagnoses:

ICD9_CODE
4019     20703
4280     13111
42731    12891
41401    12429
5849      9119
25000     9058
2724      8690
51881     7497
5990      6555
53081     6326
2720      5930
V053      5779
V290      5519
2859      5406
2449      4917
486       4839
2851      4552
2762      4528
496       4431
99592     3912
Name: count, dtype: int64

✅ Top 20 ICD9 codes saved: ['4019', '4280', '42731', '41401', '5849', '25000', '2724', '51881', '5990', '53081', '2720', 'V053', 'V290', '2859', '2449', '486', '2851', '2762', '496', '99592']


In [6]:
print("Loading NOTEEVENTS.csv (this may take 30 seconds)...")
notes = pd.read_csv('../../data/raw/NOTEEVENTS.csv')

print(f"\n✅ Loaded {len(notes):,} clinical notes")
print(f"Unique patients with notes: {notes['SUBJECT_ID'].nunique():,}")

print(f"\nNote categories:")
print(notes['CATEGORY'].value_counts())

print(f"\nSample note (first 500 characters):")
sample_note = notes['TEXT'].iloc[0]
print(sample_note[:500] if len(sample_note) > 500 else sample_note)

Loading NOTEEVENTS.csv (this may take 30 seconds)...


C:\Users\athar\AppData\Local\Temp\ipykernel_7204\1630886133.py:3: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('../data/raw/NOTEEVENTS.csv')



✅ Loaded 2,083,180 clinical notes
Unique patients with notes: 46,146

Note categories:
CATEGORY
Nursing/other        822497
Radiology            522279
Nursing              223556
ECG                  209051
Physician            141624
Discharge summary     59652
Echo                  45794
Respiratory           31739
Nutrition              9418
General                8301
Rehab Services         5431
Social Work            2670
Case Management         967
Pharmacy                103
Consult                  98
Name: count, dtype: int64

Sample note (first 500 characters):
Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]


Service:
ADDENDUM:

RADIOLOGIC STUDIES:  Radiologic studies also included a chest
CT, which confirmed cavitary lesions in the left lung apex
consistent with infectious process/tuberculosis.  This also
moderate-sized left pleural effusion.

HEAD CT:  Head CT showed no intracranial hemorrhage or mass
effect, but old infarction consistent with past

In [7]:
print("=" * 60)
print("📊 MIMIC-III DATA SUMMARY")
print("=" * 60)

print(f"\n👥 PATIENTS:")
print(f"   Total patients: {len(patients):,}")
print(f"   Male: {(patients['GENDER'] == 'M').sum():,}")
print(f"   Female: {(patients['GENDER'] == 'F').sum():,}")

print(f"\n🏥 DIAGNOSES:")
print(f"   Total diagnosis records: {len(diagnoses):,}")
print(f"   Unique ICD9 codes: {diagnoses['ICD9_CODE'].nunique():,}")
print(f"   Avg diagnoses per patient: {len(diagnoses) / len(patients):.1f}")


print(f"\n📝 CLINICAL NOTES:")
print(f"   Total notes: {len(notes):,}")
print(f"   Patients with notes: {notes['SUBJECT_ID'].nunique():,}")
print(f"   Coverage: {notes['SUBJECT_ID'].nunique() / len(patients) * 100:.1f}%")

print(f"\n✅ Data looks good! Ready for preprocessing.")
print("=" * 60)

📊 MIMIC-III DATA SUMMARY

👥 PATIENTS:
   Total patients: 46,520
   Male: 26,121
   Female: 20,399

🏥 DIAGNOSES:
   Total diagnosis records: 651,047
   Unique ICD9 codes: 6,984
   Avg diagnoses per patient: 14.0

📝 CLINICAL NOTES:
   Total notes: 2,083,180
   Patients with notes: 46,146
   Coverage: 99.2%

✅ Data looks good! Ready for preprocessing.


In [8]:
import os

# Create missing folders
os.makedirs('../../results', exist_ok=True)
os.makedirs('../../data/processed', exist_ok=True)

print("✅ Folders created!")


✅ Folders created!


In [9]:
# TARGET DIAGNOSES FOR MULTI-AGENT SYSTEM
# Focus on ACUTE ICU conditions (not chronic diseases)

target_diagnoses = {
    # Sepsis
    '99592': 'Severe sepsis',
    '99591': 'Sepsis (unspecified)',
    '78552': 'Septic shock',
    
    # Pneumonia (multiple codes starting with 48x)
    '486': 'Pneumonia (organism unspecified)',
    '48241': 'Methicillin susceptible pneumonia',
    '4829': 'Bacterial pneumonia',
    '485': 'Bronchopneumonia',
    
    # Respiratory Failure
    '51881': 'Acute respiratory failure',
    '51884': 'Acute and chronic respiratory failure',
    
    # Acute Kidney Injury
    '5849': 'Acute kidney failure (unspecified)',
    '5851': 'Acute kidney failure with lesion of tubular necrosis',
    
    # Heart Failure (acute decompensation)
    '4280': 'Congestive heart failure',
    '42821': 'Acute systolic heart failure',
    '42823': 'Acute on chronic systolic heart failure',
    
    # Cardiac
    '42731': 'Atrial fibrillation',
    '41401': 'Coronary atherosclerosis',
    
    # Other Acute Conditions
    '2859': 'Anemia (unspecified)',
    '5070': 'Acute pancreatitis',
}

print("🎯 Selected Target Diagnoses:")
print("=" * 60)
for code, name in target_diagnoses.items():
    count = len(diagnoses[diagnoses['ICD9_CODE'] == code])
    print(f"{code:8} - {name:40} ({count:,} patients)")
print("=" * 60)

# Count total unique patients with any target diagnosis
target_codes = list(target_diagnoses.keys())
target_patients = diagnoses[diagnoses['ICD9_CODE'].isin(target_codes)]['SUBJECT_ID'].unique()

print(f"\n✅ Total unique patients with target diagnoses: {len(target_patients):,}")
print(f"   This is {len(target_patients)/len(patients)*100:.1f}% of all patients")

🎯 Selected Target Diagnoses:
99592    - Severe sepsis                            (3,912 patients)
99591    - Sepsis (unspecified)                     (1,272 patients)
78552    - Septic shock                             (2,586 patients)
486      - Pneumonia (organism unspecified)         (4,839 patients)
48241    - Methicillin susceptible pneumonia        (789 patients)
4829     - Bacterial pneumonia                      (227 patients)
485      - Bronchopneumonia                         (59 patients)
51881    - Acute respiratory failure                (7,497 patients)
51884    - Acute and chronic respiratory failure    (684 patients)
5849     - Acute kidney failure (unspecified)       (9,119 patients)
5851     - Acute kidney failure with lesion of tubular necrosis (13 patients)
4280     - Congestive heart failure                 (13,111 patients)
42821    - Acute systolic heart failure             (492 patients)
42823    - Acute on chronic systolic heart failure  (1,143 patients)
42731 

In [10]:
# Group diagnoses into disease categories
def categorize_diagnosis(icd9_code):
    """Map ICD9 codes to disease categories"""
    code = str(icd9_code)
    
    # Sepsis
    if code in ['99592', '99591', '78552', '0389', '99859']:
        return 'SEPSIS'
    
    # Pneumonia (all 48x codes)
    elif code.startswith('48') or code in ['485', '486']:
        return 'PNEUMONIA'
    
    # Respiratory Failure (518xx)
    elif code.startswith('518'):
        return 'RESPIRATORY_FAILURE'
    
    # Acute Kidney Injury (584x, 585x)
    elif code.startswith('584') or code.startswith('585'):
        return 'ACUTE_KIDNEY_INJURY'
    
    # Heart Failure (428x)
    elif code.startswith('428'):
        return 'HEART_FAILURE'
    
    # Atrial Fibrillation
    elif code.startswith('4273'):
        return 'ATRIAL_FIBRILLATION'
    
    # Coronary Artery Disease (414x)
    elif code.startswith('414'):
        return 'CORONARY_ARTERY_DISEASE'
    
    # Anemia (285x)
    elif code.startswith('285'):
        return 'ANEMIA'
    
    # Pancreatitis (577x)
    elif code.startswith('577') or code.startswith('507'):
        return 'PANCREATITIS'
    
    else:
        return 'OTHER'

# Add disease category to diagnoses
diagnoses['DISEASE_CATEGORY'] = diagnoses['ICD9_CODE'].apply(categorize_diagnosis)

# Count patients per category
print("📊 Patients by Disease Category:")
print("=" * 60)
disease_counts = {}
for category in ['SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 'ACUTE_KIDNEY_INJURY', 
                 'HEART_FAILURE', 'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
                 'ANEMIA', 'PANCREATITIS']:
    patients_with_disease = diagnoses[diagnoses['DISEASE_CATEGORY'] == category]['SUBJECT_ID'].nunique()
    disease_counts[category] = patients_with_disease
    print(f"{category:30} {patients_with_disease:,} patients")

print("=" * 60)
print(f"Total: {sum(disease_counts.values()):,} (note: patients may have multiple conditions)")

📊 Patients by Disease Category:
SEPSIS                         5,865 patients
PNEUMONIA                      6,531 patients
RESPIRATORY_FAILURE            11,363 patients
ACUTE_KIDNEY_INJURY            11,514 patients
HEART_FAILURE                  10,154 patients
ATRIAL_FIBRILLATION            10,552 patients
CORONARY_ARTERY_DISEASE        11,926 patients
ANEMIA                         10,631 patients
PANCREATITIS                   4,538 patients
Total: 83,074 (note: patients may have multiple conditions)


In [11]:
# Get patients with at least one target diagnosis
target_categories = ['SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 'ACUTE_KIDNEY_INJURY', 
                     'HEART_FAILURE', 'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
                     'ANEMIA', 'PANCREATITIS']

target_patient_ids = diagnoses[
    diagnoses['DISEASE_CATEGORY'].isin(target_categories)
]['SUBJECT_ID'].unique()

print(f"🎯 Selected Cohort:")
print(f"   Patients with target conditions: {len(target_patient_ids):,}")
print(f"   Percentage of total: {len(target_patient_ids)/len(patients)*100:.1f}%")

# Get primary diagnosis for each patient (SEQ_NUM = 1)
patient_primary_dx = diagnoses[
    (diagnoses['SUBJECT_ID'].isin(target_patient_ids)) &
    (diagnoses['SEQ_NUM'] == 1)
][['SUBJECT_ID', 'ICD9_CODE', 'DISEASE_CATEGORY']]

print(f"\n📋 Primary Diagnosis Distribution:")
print(patient_primary_dx['DISEASE_CATEGORY'].value_counts())

# Save target patient list
import os
os.makedirs('../../data/processed', exist_ok=True)

patient_primary_dx.to_csv('../../data/processed/target_patients.csv', index=False)
print(f"\n✅ Saved {len(patient_primary_dx):,} patients to target_patients.csv")

🎯 Selected Cohort:
   Patients with target conditions: 31,166
   Percentage of total: 67.0%

📋 Primary Diagnosis Distribution:
DISEASE_CATEGORY
OTHER                      30415
CORONARY_ARTERY_DISEASE     3576
SEPSIS                      2395
RESPIRATORY_FAILURE         1563
HEART_FAILURE               1488
PNEUMONIA                   1137
PANCREATITIS                1101
ACUTE_KIDNEY_INJURY          681
ATRIAL_FIBRILLATION          382
ANEMIA                        44
Name: count, dtype: int64

✅ Saved 42,782 patients to target_patients.csv


In [12]:
# CREATE MULTI-LABEL DATASET
# For each patient, identify ALL target diseases they have (not just primary)

print("Creating multi-label dataset...")

# Get all target patients (those with ANY target condition)
target_patient_ids_list = list(target_patient_ids)

# For each patient, create binary labels for each disease
disease_categories = ['SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 'ACUTE_KIDNEY_INJURY', 
                     'HEART_FAILURE', 'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
                     'ANEMIA', 'PANCREATITIS']

# Create labels dataframe
patient_labels = pd.DataFrame({'SUBJECT_ID': target_patient_ids_list})

for disease in disease_categories:
    # Get patients with this disease (ANY sequence number, not just primary)
    patients_with_disease = diagnoses[
        (diagnoses['SUBJECT_ID'].isin(target_patient_ids_list)) &
        (diagnoses['DISEASE_CATEGORY'] == disease)
    ]['SUBJECT_ID'].unique()
    
    # Create binary label
    patient_labels[disease] = patient_labels['SUBJECT_ID'].isin(patients_with_disease).astype(int)

print(f"\n✅ Created multi-label dataset: {len(patient_labels):,} patients")
print(f"\nLabel distribution:")
for disease in disease_categories:
    count = patient_labels[disease].sum()
    percentage = count / len(patient_labels) * 100
    print(f"   {disease:30} {count:5,} patients ({percentage:5.1f}%)")

# Check how many diseases per patient
patient_labels['NUM_DISEASES'] = patient_labels[disease_categories].sum(axis=1)
print(f"\n📊 Diseases per patient:")
print(patient_labels['NUM_DISEASES'].value_counts().sort_index())

# Save multi-label dataset
patient_labels.to_csv('../../data/processed/patient_multilabels.csv', index=False)
print(f"\n✅ Saved multi-label dataset to patient_multilabels.csv")

Creating multi-label dataset...

✅ Created multi-label dataset: 31,166 patients

Label distribution:
   SEPSIS                         5,865 patients ( 18.8%)
   PNEUMONIA                      6,531 patients ( 21.0%)
   RESPIRATORY_FAILURE            11,363 patients ( 36.5%)
   ACUTE_KIDNEY_INJURY            11,514 patients ( 36.9%)
   HEART_FAILURE                  10,154 patients ( 32.6%)
   ATRIAL_FIBRILLATION            10,552 patients ( 33.9%)
   CORONARY_ARTERY_DISEASE        11,926 patients ( 38.3%)
   ANEMIA                         10,631 patients ( 34.1%)
   PANCREATITIS                   4,538 patients ( 14.6%)

📊 Diseases per patient:
NUM_DISEASES
1    9374
2    7698
3    5816
4    3821
5    2341
6    1272
7     571
8     219
9      54
Name: count, dtype: int64

✅ Saved multi-label dataset to patient_multilabels.csv


In [13]:
from sklearn.model_selection import train_test_split

# Load multi-label dataset
patient_labels = pd.read_csv('../../data/processed/patient_multilabels.csv')

print(f"Creating train/val/test split...")
print(f"Total patients: {len(patient_labels):,}")

# For stratification, we'll use the PRIMARY disease (most prevalent one per patient)
# This ensures balanced splits
def get_primary_disease(row):
    """Get the most prevalent disease for this patient (for stratification)"""
    disease_cols = ['SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 'ACUTE_KIDNEY_INJURY', 
                   'HEART_FAILURE', 'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
                   'ANEMIA', 'PANCREATITIS']
    
    for disease in disease_cols:
        if row[disease] == 1:
            return disease
    return 'NONE'

patient_labels['STRATIFY_DISEASE'] = patient_labels.apply(get_primary_disease, axis=1)

print(f"\nStratification disease distribution:")
print(patient_labels['STRATIFY_DISEASE'].value_counts())

# First split: 70% train, 30% temp (for val+test)
train_ids, temp_ids = train_test_split(
    patient_labels['SUBJECT_ID'].values,
    test_size=0.30,
    random_state=42,
    stratify=patient_labels['STRATIFY_DISEASE'].values
)

# Get stratify labels for temp set
temp_labels = patient_labels[patient_labels['SUBJECT_ID'].isin(temp_ids)]

# Second split: 50% val, 50% test (from the 30%)
val_ids, test_ids = train_test_split(
    temp_labels['SUBJECT_ID'].values,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels['STRATIFY_DISEASE'].values
)

print(f"\n📊 Split Sizes:")
print(f"   Train: {len(train_ids):,} patients (70%)")
print(f"   Val:   {len(val_ids):,} patients (15%)")
print(f"   Test:  {len(test_ids):,} patients (15%)")

# Save splits
pd.DataFrame({'SUBJECT_ID': train_ids}).to_csv('../../data/processed/train_patients.csv', index=False)
pd.DataFrame({'SUBJECT_ID': val_ids}).to_csv('../../data/processed/val_patients.csv', index=False)
pd.DataFrame({'SUBJECT_ID': test_ids}).to_csv('../../data/processed/test_patients.csv', index=False)

print(f"\n✅ Saved train/val/test splits!")

Creating train/val/test split...
Total patients: 31,166

Stratification disease distribution:
STRATIFY_DISEASE
SEPSIS                     5865
RESPIRATORY_FAILURE        5348
PNEUMONIA                  4367
ACUTE_KIDNEY_INJURY        4299
CORONARY_ARTERY_DISEASE    3454
ATRIAL_FIBRILLATION        2869
HEART_FAILURE              2745
ANEMIA                     1750
PANCREATITIS                469
Name: count, dtype: int64

📊 Split Sizes:
   Train: 21,816 patients (70%)
   Val:   4,675 patients (15%)
   Test:  4,675 patients (15%)

✅ Saved train/val/test splits!
